# Chapter 22 — Internal Signals

**Book alignment:** Debugging AI From First Principles, Chapter 22

**Question this notebook isolates:** At the token where the citation flips, the logprob
dips hard. Does the locate-then-probe discipline — the dip *nominates* a token, a
signal-nominated boundary probe *convicts* — separate **H1** (genuine competition, the
nominated probe flips the outcome) from **H2/H3** (signal mirage, an *un-nominated*
intervention is what flips it)?

In [ ]:
import numpy as np

# a completion as (token, logprob) pairs; the veer is where the citation turns wrong
COMPLETION = [
    ("Under", -0.22), ("section", -0.18), ("4", -0.31), (",", -0.12),
    ("the", -0.20), ("general", -6.2), ("policy", -0.4), ("applies", -0.6),
]
RUNNING_MEAN = float(np.mean([lp for _, lp in COMPLETION[:5]]))

def outcome(*, force_citation=False, remove_general_policy=False, reorder_history=False, trial):
    """Deterministic per trial. The answer is correct iff the 4.2 citation is present + surfaced."""
    rng = np.random.default_rng(trial)
    if force_citation:
        return rng.random() < 0.9              # nominated probe: force the right prefix
    if remove_general_policy:
        return rng.random() < 0.8              # nominated probe: remove the attended rival
    if reorder_history:
        return rng.random() < 0.2              # UN-nominated control: shuffle unrelated history
    return rng.random() < 0.2                  # baseline

## 1. Nominate — the dip locates a candidate, it does not explain anything

In [ ]:
veer_i = int(np.argmin([lp for _, lp in COMPLETION]))
veer_tok, veer_lp = COMPLETION[veer_i]
print(f"veer token: {veer_tok!r}  logprob {veer_lp}  vs running mean {RUNNING_MEAN:.2f}")
assert veer_lp < RUNNING_MEAN - 3
print("NOMINATION (in writing): the veer is at 'general'; attention illustration points at the")
print("general-policy paragraph. neither is a cause yet - both become boundary probes.")

## 2. Convict — signal-nominated probes vs an un-nominated control

In [ ]:
TRIALS = range(5)
force   = [outcome(force_citation=True, trial=t) for t in TRIALS]
remove  = [outcome(remove_general_policy=True, trial=t) for t in TRIALS]
control = [outcome(reorder_history=True, trial=t) for t in TRIALS]
base    = [outcome(trial=t) for t in TRIALS]

print(f"baseline           : {sum(base)}/5")
print(f"probe-A force-cite  : {sum(force)}/5   (signal-nominated)")
print(f"probe-A remove-rival: {sum(remove)}/5  (signal-nominated)")
print(f"probe-B reorder-hist: {sum(control)}/5 (UN-nominated control)")

assert sum(force) >= 4 and sum(remove) >= 4       # nominated probes flip it
assert sum(control) <= 1                          # the control does not
print("\nnominated probes flip, control flat -> H1: the dip marked genuine competition (located, not explained)")

## 3. Agreement on one run earns the signal no credit

In [ ]:
# attention 'agreed' with the winning probe here - but that is one run, not validation
attention_agreed_this_run = True
signal_is_validated = False        # would require the probe to confirm the signal's forecast across runs+revisions
assert attention_agreed_this_run and not signal_is_validated
print("a heatmap that matches the probe once is a coincidence with formatting")
print("route high-entropy runs to a REVIEW QUEUE (triage) - never label them wrong from the signal (verdict)")

## What we earned

Interior signals are body language: a logprob dip *locates* the token to probe first,
attention *illustrates* the segment to move first — and neither testifies. Conviction came
only from a signal-nominated boundary probe (force the citation / remove the attended
rival) that flipped the outcome as forecast, with an un-nominated control that stayed flat
to keep it honest. The distributional signals (semantic entropy, self-consistency) have
real *detection* value; single-run dips do not even reach that. Route by uncertainty;
decide by probe.

**Notebook 23 / Chapter 23** turns a located, probed behavior into an asset that survives
the next model revision: the behavioral diff.